# Business Issue: 법인 KYC 심사

**문제**: 법인 등록·대표자·실소유자(UBO) 증빙과 제재·PEP 스크리닝은 여러 시스템에 흩어져 있습니다. 누락을 놓치지 않으면서도 자동으로 계정을 개설하지 않는 심사 흐름이 필요합니다.

이 노트북의 데이터는 합성이며 PII를 포함하지 않습니다. Harness는 Business Agent 라우팅과 근거 패킷 형식을 제공하고, 정책 판정은 투명한 코드로 수행합니다.

In [ ]:
# 합성 KYC snapshot과 정책 게이트입니다. 외부 고객 시스템을 호출하지 않습니다.
case = {"request": "신규 법인 고객의 KYC를 검토해줘.", "customer": {"legal_name": "데모에너지솔루션", "customer_id": "DEMO-C-1042"}, "verification": {"corporate_registration_verified": True, "representative_identity_verified": True, "ultimate_beneficial_owner_verified": False, "sanctions_screening": "clear", "pep_screening": "possible_match"}}
def assess_kyc(data):
    v = data["verification"]; flags = []
    if v["sanctions_screening"] == "confirmed_match": decision, flags = "block", ["confirmed_sanctions_match"]
    else:
        if not v["ultimate_beneficial_owner_verified"]: flags.append("unverified_ubo")
        if v["pep_screening"] == "possible_match": flags.append("possible_pep_match")
        decision = "enhanced_due_diligence" if flags else "eligible_for_compliance_approval"
    return {"decision": decision, "risk_flags": flags, "account_activation_allowed": False, "action": "compliance_approval_required", "write_performed": False}


## 1. 읽기 전용 심사 근거

실제 구현에서는 CRM, 법인등기 확인, 제재·PEP 스크리닝 도구를 **읽기 전용**으로 호출합니다. fixture는 그 결과를 축약해 재현합니다.

In [ ]:
print(case["request"]); print("고객:", case["customer"]["legal_name"], case["customer"]["customer_id"])
for check, value in case["verification"].items(): print(f"- {check}: {value}")


## 2. Harness → KYC 정책 게이트

Harness reranker가 Business Agent를 선택하면, 증거 패킷은 사실과 한계를 분리합니다. 정책 게이트는 확정 제재 매칭을 차단하고, UBO 미확인 또는 PEP 가능 매칭은 강화심사(EDD)로 이관합니다.

In [ ]:
result = assess_kyc(case)
print("선택 specialist: business-agent (KYC 업무 요청)"); print("판정:", result["decision"]); print("위험 플래그:", ", ".join(result["risk_flags"]))


## 3. Human-in-the-loop

KYC는 자동 계정 개설로 이어지지 않습니다. `possible_match`는 확정 매칭이 아니므로 모델이 제재 대상이라고 단정하지 않고, 준법감시인이 추가 자료와 매칭 근거를 검토합니다.

In [ ]:
print("상태:", result["action"]); print("계정 활성화 허용:", result["account_activation_allowed"]); print("실제 write 수행:", result["write_performed"])
